In [1]:
# import libs
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [2]:
import os

# 1. Set the Kaggle env, which I just use for API instead of downloading it, due to I am working on Colab
# And I deleted it so I can submit my HW
os.environ['KAGGLE_USERNAME'] = "Jisheng Jiang"
os.environ['KAGGLE_KEY'] = "KGAT_cdf019d35324ab87f9ca7969ed34607a"

# 2. Downloading
!kaggle datasets download -d thedevastator/youtube-video-and-channel-analytics

# 3. Unzip it
!unzip youtube-video-and-channel-analytics.zip

Dataset URL: https://www.kaggle.com/datasets/thedevastator/youtube-video-and-channel-analytics
License(s): other
  0% 0.00/81.6M [00:00<?, ?B/s]
100% 81.6M/81.6M [00:00<00:00, 1.02GB/s]
Archive:  youtube-video-and-channel-analytics.zip
  inflating: YouTubeDataset_withChannelElapsed.csv  


In [3]:
# Make sure Dataset exist
df = pd.read_csv('YouTubeDataset_withChannelElapsed.csv')

df.head()

,index,totalviews/channelelapsedtime,channelId,videoCategoryId,channelViewCount,likes/subscriber,views/subscribers,videoCount,subscriberCount,videoId,dislikes/views,channelelapsedtime,comments/subscriber,likes/views,channelCommentCount,videoViewCount,likes/dislikes,comments/views,totvideos/videocount,elapsedtime,videoLikeCount,videoDislikeCount,dislikes/subscriber,totviews/totsubs,views/elapsedtime,videoPublished,VideoCommentCount
0,0,0.165199,UCdzU3DSGzyWzN2118yd9X9g,22,14654,0.555556,95.111111,30,18,--DwgB78t-c,0.000584,88705,0.000000,0.005841,1,1712,-2.0,0.000000,488.466667,50040,10,1,0.055556,814.111111,0.034213,2012-01-19T18:38:28.000Z,0
1,1,1.133820,UC0UnhAG47DRyVZGVcbhAXhQ,10,105909,0.239130,59.326087,51,184,--NZRkXBV7k,0.000275,93409,0.010870,0.004031,8,10916,-2.0,0.000183,2076.647059,22080,44,3,0.016304,575.592391,0.494384,2015-03-30T04:04:40.000Z,2
2,2,0.668120,UCXjtAvK5P3wXBGh0vbGylzg,27,48265,0.023669,10.289941,72,338,--hoQ2sGG4M,0.000288,72240,0.005917,0.002300,5,3478,-2.0,0.000575,670.347222,71544,8,1,0.002959,142.795858,0.048613,2009-08-07T06:51:10.000Z,2
3,3,25.653505,UCeKHMeUlcLNPLCLUfZUQI2w,26,2116722,0.007301,0.884178,172,22051,--sBoaqBlzA,0.000308,82512,0.000453,0.008258,74,19497,-2.0,0.000513,12306.523256,54096,161,6,0.000272,95.992109,0.360415,2011-08-04T01:07:38.000Z,10
4,4,52.773778,UCNWPDyaWf2eAHnofFLSnEMg,20,1649075,0.004545,10.004545,2777,220,--7h1S4neDM,0.000000,31248,0.000000,0.000454,0,2201,-2.0,0.000000,593.833273,30120,1,0,0.000000,7495.795455,0.073074,2014-04-29T15:44:44.000Z,0


In [4]:
# Ensure that each videoId in the dataset is unique
# Max is used here because, considering that the video will update data, the maximum value is directly employed to ensure it remains the latest.
df_unique = df.groupby('videoId').max().reset_index()

# Just to make sure the above code is indeed useful.
print(f"ORIGINAL: {len(df)}")
print(f"After deletion: {len(df_unique)}")

# Q1: Average number of likes per video
# Perform calculations using the after deletion `df_unique`, retaining integer values.
avg_likes = df_unique['videoLikeCount'].mean()

# The reason for splitting it into two parts is that Q1 did not specify the need for rounding
# But the rounded numbers are more reasonable.
# The unrounded numbers will serve as the basis for subsequent calculations.
round_avg_likes = round(avg_likes)

print(f"\nQuestion 1 answer is: {round_avg_likes}")

ORIGINAL: 575610
After deletion: 555627

Question 1 answer is: 293


In [5]:
# Q2. Compute the engagement rate
# (likes + dislikes + comments) / views
df_unique = df_unique[df_unique['videoViewCount'] > 0].copy()

df_unique['engagement_rate'] = (
    df_unique['videoLikeCount'] +
    df_unique['videoDislikeCount'] +
    df_unique['VideoCommentCount']
) / df_unique['videoViewCount']

# average engagement rate(round to nearest 4 decimal points)
avg_engagement_rate = round(df_unique['engagement_rate'].mean(), 4)
print(f"Question 2 (Average Engagement Rate): {avg_engagement_rate}")

Question 2 (Average Engagement Rate): 0.0084


In [6]:
# Q3
# Define
KB_FACTOR = 1024
MB_FACTOR = 1024 * 1024

# 1. metadata (MB)
metadata_size_mb = 500 / MB_FACTOR

# 2. thumbnail (MB)
thumbnail_size_mb = 200 / KB_FACTOR

# 3. combined storage size
total_size_mb = metadata_size_mb + thumbnail_size_mb

storage_per_video_mb = round(total_size_mb, 4)

print(f"The size of a video's metadata: {metadata_size_mb:.4f} MB")
print(f"The size of a video's thumbnail: {thumbnail_size_mb:.4f} MB")
print(f"Question 3 (Storage per Video MB): {storage_per_video_mb}")

The size of a video's metadata: 0.0005 MB
The size of a video's thumbnail: 0.1953 MB
Question 3 (Storage per Video MB): 0.1958


In [7]:
# define
TOTAL_MINUTES_5Y = 5 * 365 * 24 * 60

# --- Question 4: total data transfer ---
total_engagements = (
    df_unique['videoLikeCount'] +
    df_unique['videoDislikeCount'] +
    df_unique['VideoCommentCount']
).sum()

total_engagement_transfer_mb = round(total_engagements * storage_per_video_mb)

print(f"Question 4 (Total Transfer MB): {total_engagement_transfer_mb}")

Question 4 (Total Transfer MB): 38166792


In [8]:
# --- Question 5: number of views per minute ---
total_views = df_unique['videoViewCount'].sum()
round_total_views = round(total_views / TOTAL_MINUTES_5Y)

print(f"Question 5's answer is: {round_total_views}")

Question 5's answer is: 12473


In [9]:
# --- Question 6: data accessed per minute ---
# Data access generated per minute of viewing
data_accessed_per_minute_mb = round(round_total_views * storage_per_video_mb)

print(f"Question 6 (Data per Minute MB): {data_accessed_per_minute_mb}")

Question 6 (Data per Minute MB): 2442
